In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
mkdir -p "$work"

repo="$work/CryoAtom"
if [ ! -d "$repo" ]; then
    git clone https://github.com/YangLab-SDU/CryoAtom.git "$repo"
fi
git -C "$repo" checkout --quiet --detach v1.0.0

if [ ! -s "$repo/CryoAtom/checkpoint/CryNet.pth" ] || [ ! -s "$repo/CryoAtom/checkpoint/SimpleUnet.pth" ]; then
    curl -fLo "$work/cryoatom-weights.zip" https://yanglab.qd.sdu.edu.cn/CryoAtom/download/checkpoints_v1.0.zip
    python3 -m zipfile -e "$work/cryoatom-weights.zip" "$repo"
    rm "$work/cryoatom-weights.zip"
fi
case "$(uname -s):$(uname -m)" in
Darwin:*)
    export PATH="$work/bin:$PATH"
    if ! command -v uv >/dev/null; then
        curl -fsSL https://astral.sh/uv/install.sh | UV_UNMANAGED_INSTALL="$work/bin" sh
    fi
    cd "$work"
    env="$work/CryoAtom-env"
    if [ ! -x "$env/bin/python" ]; then uv venv --python 3.11 "$env"; fi
    uv pip install --python "$env/bin/python" torch==2.5.1 numpy==1.26.4 \
        einops==0.6.0 scipy biopython matplotlib mrcfile pandas fair-esm==2.0.0 pyhmmer==0.7.1 openpyxl tqdm
    ;;
Linux:x86_64)
    conda="$(command -v conda || printf '%s' "$work/miniforge/bin/conda")"
    if [ ! -x "$conda" ]; then
        curl -fLo "$work/miniforge.sh" https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh
        bash "$work/miniforge.sh" -b -p "$work/miniforge"
    fi

    env="$work/CryoAtom-env"
    CONDA_CHANNEL_PRIORITY=flexible "$conda" env update -p "$env" -f "$repo/linux.yml"
    ;;
*)
    echo "CryoAtom requires macOS or x86-64 Linux."
    exit 1
    ;;
esac


In [ ]:
%%bash
set -eo pipefail
# Build both targets
root="$PWD"
work="$root/archives/cryo-benchmark"

method=CryoAtom
export MPLBACKEND=Agg TORCH_HOME="$work/torch-cache"
cd "$work/CryoAtom"
if [ "$(uname -s)" = Darwin ]; then
    python=("$work/CryoAtom-env/bin/python")
    device=cpu
else
    conda="${CONDA_EXE:-$(command -v conda || printf '%s' "$work/miniforge/bin/conda")}"
    python=("$conda" run -p "$work/CryoAtom-env" --no-capture-output python)
    device=cuda:0
fi

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"

    time "${python[@]}" -m CryoAtom build -v "${map[0]}" -s "$fasta" -o "$out" -d "$device"
done


In [ ]:
%%bash
set -eo pipefail
# Score both targets
root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="CryoAtom"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$out/$target.cif"
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done
